In [ ]:
import os
import librosa
import librosa.display
import json 
import importlib
import formExtractor as fem
importlib.reload(fem)
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
np.version.version

In [ ]:
def load_audio_files(path):
    audio_files = []
    for root, dirs, files in os.walk(path):
        for file in files:
            if file.endswith(".mp3"):
                audio_files.append(os.path.join(root, file))
    return audio_files

In [ ]:
path = '/mnt/Aimir_HD/' 
collection = 'lastfm'
extra = 'audio'
song_files = os.path.join(path, collection, extra)
files = load_audio_files(song_files)
print(len(files))

In [ ]:
id = 3
song = files[id]

#song = '/home/laura/aimir/boomy/audio/14065870.mp3'
id_file = song.split('/')[-1].split('.')[0]
song

In [ ]:
# print metadata

md = os.path.join(path, collection, 'metadata', id_file + '.json')
print(md)

with open(md) as json_file:
    data = json.load(json_file)
    
    # Calculate the maximum key length
    max_key_length = max(len(key) for key in data.keys())
    
    # Print each line with keys aligned
    for key in data:
        print(f"{key:{max_key_length}} : {data[key]}")

In [ ]:
formData = fem.formExtractor()

y, sr = librosa.load(song)

In [ ]:
# Extract chroma features
chroma = librosa.feature.chroma_stft(y=y, sr=sr)
# Aggregate chroma features by averaging over time
chroma_mean = np.mean(chroma, axis=1)

In [ ]:
# Krumhansl-Schmuckler key profiles
major_profile = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52,
                          5.19, 2.39, 3.66, 2.29, 2.88])

minor_profile = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54,
                          4.75, 3.98, 2.69, 3.34, 3.17])

In [ ]:
# Initialize lists to store correlation scores
correlations = []

# Iterate over all 12 keys
keys = ['C', 'C#', 'D', 'D#', 'E', 'F',
        'F#', 'G', 'G#', 'A', 'A#', 'B']
modes = ['major', 'minor']

for i, key in enumerate(keys):
    # Rotate the key profiles to match the current key
    rotated_major = np.roll(major_profile, i)
    rotated_minor = np.roll(minor_profile, i)
    
    # Compute correlations
    corr_major = np.corrcoef(chroma_mean, rotated_major)[0, 1]
    corr_minor = np.corrcoef(chroma_mean, rotated_minor)[0, 1]
    
    correlations.append((f"{key} major", corr_major))
    correlations.append((f"{key} minor", corr_minor))

# Sort keys by correlation score
correlations.sort(key=lambda x: x[1], reverse=True)

# Print top 5 key suggestions
print("Top 5 key detections:")
for key, score in correlations[:5]:
    print(f"{key}: {score:.4f}")

#get the first tonality in the list
tonality = correlations[0][0]


In [ ]:
C = formData.amplitud_to_db(y, sr, False)

In [ ]:
#Example 1, calculate all the features and save them

#path = '/home/laura/aimir/suno/form/'
#K = 2
#data_dict = formData.getFormAndSave(K, song, id_file, path)

# Call the plotSpectrogram function
#plotMe.plotSpectrogram(sr, C, chords, bars, bound_frames, new_bound_segs, BINS_PER_OCTAVE, 40, 5)

#plotMe.plotWaveform(y, data_dict['sr'], data_dict['chords'], data_dict['bars'], data_dict['bound_frames'], data_dict['bound_segs'], 40, 5) #last two numbers are the size of the plot in inches

#from IPython.display import Audio
#Audio(song)

In [ ]:
Csync, beats, beat_times = formData.sync(y, sr, C)

In [ ]:
import formExtractor as fem

# Recreate the instance after reloading
formData = fem.formExtractor()

K = 5 #If we want k clusters, use the first k normalized eigenvectors.
min_duration = 3  # Minimum segment duration in seconds

#bound_frames, bound_segs = formData.laplacian(y, sr, C, Csync, beats, beat_times, K, True, 0.0) #True to plot the laplacian

bound_frames, bound_segs = formData.laplacian_2(y, sr, C, Csync, beats, beat_times, K, min_duration=min_duration)

In [ ]:
# get triads
chord_progression = formData.getChords(song)
bars = formData.getBars(song)

print(chord_progression)
#print the chords and the bars
# for chord in chord_progression:
#     print(chord)

In [ ]:
# Correct sharp or flat tonality
# Import your Transposition class
from transposition import Transposition

# Initialize the Transposition object
transposer = Transposition()

# Use the get_alterations_scales method to get alterations
corrected_tonality, alterations, scale = transposer.get_alterations_scales(tonality)

# Print the output
print(f"Tonality: {corrected_tonality}")
print(f"Alterations: {alterations}")
print(f"Scale: {scale}")

tonality = corrected_tonality

In [ ]:
import triadExtractor as te
triads = te.TriadExtractor(hop_length=1024, scale=scale)
chords = triads.extract_chords(song, threshold=0.05, check_on_beat=True)

In [ ]:
import formExtractor as fem
# Recreate the instance after reloading
formData = fem.formExtractor()
K = 5 #If we want k clusters, use the first k normalized eigenvectors.
min_duration = 3 # Minimum segment duration in seconds
bound_frames, bound_segs = formData.laplacian_2(y, sr, C, Csync, beats, beat_times, K, min_duration=min_duration)

# Convert bound_frames to bound_times - THIS IS THE MISSING LINE
bound_times = librosa.frames_to_time(bound_frames, sr=sr)

# get triads
chord_progression = formData.getChords(song)
bars = formData.getBars(song)
print(chord_progression)

# Now this should work:
data_dict = formData.populateDict(sr, chords, bars, bound_times, bound_frames, bound_segs)

In [ ]:
#The array contains a dictionary [ChordChange(chord='N', timestamp=0.371519274), ChordChange(chord='C#m', timestamp=0.464399092), print only the chord
print([chord.chord for chord in chord_progression])

In [ ]:
#print the chords only
thisChord = [chord.chord for chord in chord_progression]
thisTimestamp = [chord.timestamp for chord in chord_progression]
print(len(thisChord), len(thisTimestamp))

#align both lists
final_CP = list(zip(thisChord, thisTimestamp))
print(final_CP)

In [ ]:
#plotMe.plotSections(y, data_dict['sr'], data_dict['chords'], data_dict['bars'], data_dict['bound_frames'], data_dict['bound_segs'], 20, 5, 64) #20 and 5 are the size of the plot in inches, and 32 is the number of bars to plot

In [ ]:
#plotMe.plotChordsBars(data_dict['chords'], data_dict['bars'], data_dict['bound_frames'], data_dict['bound_segs'])

In [ ]:
#Get the tonality of the song
# Compute the chroma features using CQT
chroma_cq = librosa.feature.chroma_cqt(y=y, sr=sr)

# Sum chroma features over time to emphasize prominent pitches
chroma_vector = np.sum(chroma_cq, axis=1)
chroma_vector /= np.linalg.norm(chroma_vector)

# Modified Krumhansl-Schmuckler key profiles
major_profile = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09,
                          2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
minor_profile = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53,
                          2.54, 4.75, 3.98, 2.69, 3.34, 3.17])

# Optionally adjust profiles
# minor_profile[0] *= 1.2  # Emphasize tonic in minor

# Normalize the profiles
major_profile /= np.linalg.norm(major_profile)
minor_profile /= np.linalg.norm(minor_profile)

key_names = ['C', 'C#', 'D', 'D#', 'E', 'F',
             'F#', 'G', 'G#', 'A', 'A#', 'B']
correlations = []

for i in range(12):
    # Rotate the key profiles
    major_profile_rotated = np.roll(major_profile, i)
    minor_profile_rotated = np.roll(minor_profile, i)

    # Compute the correlation with adjusted weights
    major_corr = np.dot(chroma_vector, major_profile_rotated) * 0.9
    minor_corr = np.dot(chroma_vector, minor_profile_rotated) * 1.1  # Increase minor influence

    correlations.append({
        'key': key_names[i],
        'mode': 'major',
        'correlation': major_corr
    })
    correlations.append({
        'key': key_names[i],
        'mode': 'minor',
        'correlation': minor_corr
    })

# Find the best matching key
best_match = max(correlations, key=lambda x: x['correlation'])
tonality = f"{best_match['key']} {best_match['mode']}"
print(f"Tonality: {tonality}")

# Sort correlations in descending order
correlations_sorted = sorted(correlations, key=lambda x: x['correlation'], reverse=True)
print("Correlation scores for all keys:")
for corr in correlations_sorted:
    print(f"{corr['key']} {corr['mode']}: {corr['correlation']:.4f}")


In [ ]:
from music21 import key, harmony, roman
from IPython.display import Audio

# Define the key context explicitly as C major
mykey = tonality.split(' ')[0]
mymode = tonality.split(' ')[1]
print(mykey, mymode)
song_key = key.Key(mykey, mymode)

#print(f"Key: {song_key.tonic.name} {song_key.mode}")

#The array contains a dictionary [ChordChange(chord='N', timestamp=0.371519274), ChordChange(chord='C#m', timestamp=0.464399092), save the tuple with the function and its location in the timestamp
functional_harmony = []

for chord_name, timestamp in (chords):
    if chord_name == 'N':
        # Skip any chords labeled as N
        functional_harmony.append(('N', timestamp))
        continue
    else:   
        # Create a ChordSymbol object to correctly interpret chord qualities
        chord_symbol = harmony.ChordSymbol(chord_name)
        
        # Convert the chord symbol to a Roman numeral based on the key
        roman_numeral = roman.romanNumeralFromChord(chord_symbol, song_key)
     
        # Append the Roman numeral and timestamp to the list
        functional_harmony.append((roman_numeral.figure, timestamp))
   
print(functional_harmony)

from IPython.display import Audio
Audio(song)

In [ ]:
#get the name of folder where the data will be saved
myFolder = os.getcwd().split('src')[0]
thisPath = myFolder + collection + '/'
thisPath
#formData.saveData(data_dict, id_file, thisPath, tonality, functional_harmony)

In [ ]:
import formExtractor as fem
import importlib
importlib.reload(fem)
# Now plot
import plotMe
importlib.reload(plotMe)

# Create a formExtractor instance
formData = fem.formExtractor()

# First, you need to extract chords and beats reference data
# You can do this using your existing code or create placeholder data

# Option 1: Use your existing chord extraction code
import triadExtractor as te
from transposition import Transposition

# Get tonality (from your existing code)
chroma = librosa.feature.chroma_stft(y=y, sr=sr)
chroma_mean = np.mean(chroma, axis=1)
major_profile = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
minor_profile = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])

correlations = []
keys = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
for i, key in enumerate(keys):
    rotated_major = np.roll(major_profile, i)
    rotated_minor = np.roll(minor_profile, i)
    corr_major = np.corrcoef(chroma_mean, rotated_major)[0, 1]
    corr_minor = np.corrcoef(chroma_mean, rotated_minor)[0, 1]
    correlations.append((f"{key} major", corr_major))
    correlations.append((f"{key} minor", corr_minor))

correlations.sort(key=lambda x: x[1], reverse=True)
tonality = correlations[0][0]

# Get scale and chords
transposer = Transposition()
corrected_tonality, alterations, scale = transposer.get_alterations_scales(tonality)
triads = te.TriadExtractor(hop_length=1024, scale=scale)
chords = triads.extract_chords(song, threshold=0.05, check_on_beat=True)

# Get beats reference
tempo, beats_reference = librosa.beat.beat_track(y=y, sr=sr)

# Now run the optimized analysis with all required arguments
data_dict = formData.optimize_song_structure(song, chords, beats_reference, K=4, min_duration=3.5, plot_it=True)

# If chords are in tuple format like (chord_name, timestamp), convert them
if data_dict['chords'] and isinstance(data_dict['chords'][0], tuple):
    converted_chords = []
    for i, chord_tuple in enumerate(data_dict['chords']):
        chord_dict = {
            'beat_start': chord_tuple[1] if len(chord_tuple) > 1 else i,
            'functional_harmony': {
                'functional': chord_tuple[0] if len(chord_tuple) > 0 else 'Unknown'
            }
        }
        converted_chords.append(chord_dict)
    data_dict['chords'] = converted_chords


plotMe.plotWaveform(formData.y, data_dict['sr'], data_dict['chords'],
                   data_dict['beats'], data_dict['bound_frames'],
                   data_dict['bound_segments'], 40, 5, 'coolwarm')

In [ ]:
import os
import librosa
import importlib
import numpy as np
import matplotlib.pyplot as plt
import formExtractor as fem
import plotMe

# Reload modules after adding new functions
importlib.reload(fem)
importlib.reload(plotMe)

# Path to your audio file
song = '../samples/suno_samples/audio/2cf8e9fb-f670-4223-89a8-d6bbb83887fd.mp3'
id_file = song.split('/')[-1].split('.')[0]

# Create formExtractor instance
formData = fem.formExtractor()

# Load audio
y, sr = librosa.load(song)

# Extract beats for beats_reference
tempo, beats_reference = librosa.beat.beat_track(y=y, sr=sr)

# Define chords (since your getChords method returns empty dict, we'll use empty list)
chords = []  # or use your chord extraction code if you have it working

# Extract form structure with enhanced bar alignment
data_dict = formData.optimize_song_structure(song, chords, beats_reference, K=4, min_duration=3.5)

# First, plot the waveform with regular visualization for comparison
# Note: Check the actual keys in data_dict to make sure they match
print("Available keys in data_dict:", list(data_dict.keys()))

plotMe.plotWaveform(y, data_dict['sr'], data_dict['chords'],
                   data_dict['beats'], data_dict['bound_frames'], 
                   data_dict['bound_segments'], 40, 5, 'coolwarm')

# Play the audio for reference
from IPython.display import Audio
Audio(song)